# AI Builder via the Power Platform REST API

## Learning Objectives

By completing this notebook, you will:
1. Understand the Power Platform REST API endpoint structure for AI Builder
2. Authenticate and call AI Builder's sentiment analysis endpoint programmatically
3. Submit documents for AI Builder invoice processing and parse the response
4. Compare AI Builder's API response format with direct Azure AI Services calls
5. Build a reusable Python client for AI Builder predictions

## Why This Matters

Power Automate's no-code interface is the primary way to use AI Builder, but the underlying
predictions run through the Power Platform REST API. Understanding this API lets you:
- Call AI Builder models from Python scripts or external services
- Test model responses before wiring them into flows
- Build integrations outside of Power Automate (Azure Functions, web apps)
- Debug unexpected model outputs by inspecting the raw response structure

## Prerequisites

- Python 3.9 or higher
- `requests` library (`pip install requests`)
- A Power Platform environment with AI Builder enabled
- An Azure AD app registration with Dataverse API permissions (setup covered in cell 3)

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives([
    "Call AI Builder models from Python scripts or external services",
    "Test model responses before wiring them into flows",
    "Build integrations outside of Power Automate (Azure Functions, web apps)",
    "Debug unexpected model outputs by inspecting the raw response structure",
    "Python 3.9 or higher",
    "`requests` library (`pip install requests`)",
    "A Power Platform environment with AI Builder enabled",
    "An Azure AD app registration with Dataverse API permissions (setup covered in cell 3)"
])

## 1. Setup and Authentication

The Power Platform REST API uses Azure AD OAuth 2.0 for authentication. You need:
- A **Tenant ID**: your Azure AD tenant (found in Azure Portal > Azure Active Directory > Overview)
- A **Client ID**: from an Azure AD app registration with `Dynamics CRM` (Dataverse) permissions
- A **Client Secret**: generated from the app registration
- Your **Environment URL**: e.g. `https://yourorg.crm.dynamics.com`

The authentication flow uses the **client credentials grant** (machine-to-machine), suitable for
automated scripts that don't require a user login prompt.

```
Azure AD Token Endpoint
        |
        | POST with client_id, client_secret, scope
        |
        v
    Access Token (Bearer, expires in ~3600 seconds)
        |
        | Authorization: Bearer <token>
        |
        v
    Power Platform API Endpoint
```

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Install required packages if not already present
# Uncomment the line below on first run
# !pip install requests python-dotenv

import os
import json
import base64
import time
from typing import Any

import requests
from dotenv import load_dotenv

load_dotenv()

print("Imports successful")

In [ ]:
# Configuration — load from environment variables
# Create a .env file in this directory with these keys:
#   AZURE_TENANT_ID=your-tenant-id
#   AZURE_CLIENT_ID=your-app-client-id
#   AZURE_CLIENT_SECRET=your-app-client-secret
#   POWER_PLATFORM_ENV_URL=https://yourorg.crm.dynamics.com

TENANT_ID = os.getenv("AZURE_TENANT_ID", "")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")
ENV_URL = os.getenv("POWER_PLATFORM_ENV_URL", "").rstrip("/")

# Validate that credentials are present
missing = [name for name, val in [
    ("AZURE_TENANT_ID", TENANT_ID),
    ("AZURE_CLIENT_ID", CLIENT_ID),
    ("AZURE_CLIENT_SECRET", CLIENT_SECRET),
    ("POWER_PLATFORM_ENV_URL", ENV_URL),
] if not val]

if missing:
    print(f"WARNING: Missing environment variables: {missing}")
    print("The notebook will show request structures but cannot make live calls.")
    print("Set these in your .env file to run live calls against your environment.")
else:
    print(f"Configuration loaded. Environment: {ENV_URL}")

## 2. Obtain an Access Token

The token endpoint for Azure AD uses the format:
```
https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token
```

The scope for Power Platform (Dataverse) APIs is:
```
{environment_url}/.default
```

For example: `https://contoso.crm.dynamics.com/.default`

The token returned is a JWT that you include as a Bearer token in all subsequent API requests.
It expires after 3600 seconds (1 hour). The client below caches the token and refreshes it
automatically before it expires.

In [ ]:
# Why: Token acquisition is the foundation for all Power Platform API calls.
# The cache prevents unnecessary round trips to Azure AD on every prediction call.

class PowerPlatformClient:
    """
    Authenticated client for the Power Platform REST API.

    Handles token acquisition, caching, and renewal automatically.
    All AI Builder prediction calls go through this client.

    Parameters
    ----------
    tenant_id : str
        Azure AD tenant ID
    client_id : str
        App registration client ID with Dataverse permissions
    client_secret : str
        App registration client secret
    env_url : str
        Power Platform environment URL (e.g. https://org.crm.dynamics.com)
    """

    TOKEN_ENDPOINT = "https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    TOKEN_BUFFER_SECONDS = 300  # Refresh 5 minutes before expiry

    def __init__(
        self,
        tenant_id: str,
        client_id: str,
        client_secret: str,
        env_url: str,
    ) -> None:
        self.tenant_id = tenant_id
        self.client_id = client_id
        self.client_secret = client_secret
        self.env_url = env_url.rstrip("/")
        self._token: str | None = None
        self._token_expiry: float = 0.0

    def _acquire_token(self) -> str:
        """Request a new access token from Azure AD."""
        url = self.TOKEN_ENDPOINT.format(tenant_id=self.tenant_id)
        payload = {
            "grant_type": "client_credentials",
            "client_id": self.client_id,
            "client_secret": self.client_secret,
            # Scope targets your specific Dataverse environment
            "scope": f"{self.env_url}/.default",
        }
        response = requests.post(url, data=payload, timeout=30)
        response.raise_for_status()

        data = response.json()
        self._token = data["access_token"]
        # expires_in is in seconds; store absolute expiry time
        self._token_expiry = time.time() + data["expires_in"]
        return self._token

    @property
    def token(self) -> str:
        """Return a valid access token, refreshing if near expiry."""
        if self._token is None or time.time() >= (self._token_expiry - self.TOKEN_BUFFER_SECONDS):
            self._acquire_token()
        return self._token

    def get(self, path: str, **kwargs: Any) -> requests.Response:
        """Make an authenticated GET request to the environment API."""
        url = f"{self.env_url}/api/data/v9.2/{path}"
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Accept": "application/json",
            "OData-MaxVersion": "4.0",
            "OData-Version": "4.0",
        }
        return requests.get(url, headers=headers, timeout=60, **kwargs)

    def post(self, path: str, payload: dict, **kwargs: Any) -> requests.Response:
        """Make an authenticated POST request to the environment API."""
        url = f"{self.env_url}/api/data/v9.2/{path}"
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json",
            "Accept": "application/json",
            "OData-MaxVersion": "4.0",
            "OData-Version": "4.0",
        }
        return requests.post(url, headers=headers, json=payload, timeout=60, **kwargs)


# Instantiate the client
# If credentials are missing, the client is created but calls will fail
client = PowerPlatformClient(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    env_url=ENV_URL,
)

print("PowerPlatformClient created")
print(f"Target environment: {ENV_URL or '(not configured)'}")

## 3. Sentiment Analysis via AI Builder API

AI Builder exposes pre-built model predictions through the Dataverse Custom API endpoint:

```
POST {env_url}/api/data/v9.2/msdyn_PredictTextSentiment
```

The request body contains the text to analyze and optional language hint.
The response contains sentiment classification and per-sentiment confidence scores.

### Request Body Structure
```json
{
  "msdyn_Text": "The invoice was processed quickly and the support team was very helpful.",
  "msdyn_LanguageCode": "en"
}
```

### Response Structure
```json
{
  "msdyn_PositiveScore": 0.92,
  "msdyn_NegativeScore": 0.03,
  "msdyn_NeutralScore": 0.05,
  "msdyn_Sentiment": "positive",
  "msdyn_MixedScore": 0.00
}
```

In [ ]:
# Why: Calling sentiment analysis programmatically lets us test model behavior
# on representative samples before wiring it into a production flow.

def analyze_sentiment(client: PowerPlatformClient, text: str, language: str = "en") -> dict:
    """
    Analyze the sentiment of a text string using AI Builder's pre-built model.

    Parameters
    ----------
    client : PowerPlatformClient
        Authenticated Power Platform client
    text : str
        Text to analyze
    language : str
        BCP-47 language code (default: 'en'). Pass empty string for auto-detection.

    Returns
    -------
    dict
        Keys: sentiment (str), positive_score, negative_score, neutral_score,
              mixed_score (all float, 0.0-1.0), raw_response (dict)

    Raises
    ------
    requests.HTTPError
        If the API returns a non-2xx status code
    """
    payload = {"msdyn_Text": text}
    if language:
        payload["msdyn_LanguageCode"] = language

    response = client.post("msdyn_PredictTextSentiment", payload)
    response.raise_for_status()

    raw = response.json()

    # Normalize field names: strip the msdyn_ prefix for convenience
    return {
        "sentiment": raw.get("msdyn_Sentiment", "unknown"),
        "positive_score": raw.get("msdyn_PositiveScore", 0.0),
        "negative_score": raw.get("msdyn_NegativeScore", 0.0),
        "neutral_score": raw.get("msdyn_NeutralScore", 0.0),
        "mixed_score": raw.get("msdyn_MixedScore", 0.0),
        "raw_response": raw,
    }


# Sample texts representing the range of sentiments you might see in vendor emails
SAMPLE_TEXTS = [
    "Thank you for your prompt payment. We look forward to continuing our partnership.",
    "This invoice has been outstanding for 60 days. Please remit payment immediately or we will escalate.",
    "Attached is invoice #INV-2024-0891 for services rendered in February.",
    "We are disappointed with the delay and the lack of communication from your accounts team.",
]

# Demonstrate the request structure before making a live call
demo_payload = {"msdyn_Text": SAMPLE_TEXTS[0], "msdyn_LanguageCode": "en"}
print("Request payload structure:")
print(json.dumps(demo_payload, indent=2))
print()
print("Expected response structure:")
print(json.dumps({
    "msdyn_Sentiment": "positive",
    "msdyn_PositiveScore": 0.91,
    "msdyn_NegativeScore": 0.02,
    "msdyn_NeutralScore": 0.07,
    "msdyn_MixedScore": 0.00,
}, indent=2))

In [ ]:
# Live API call — runs only if credentials are configured
# To run: ensure .env is populated with your Power Platform credentials

if all([TENANT_ID, CLIENT_ID, CLIENT_SECRET, ENV_URL]):
    print("Running live sentiment analysis calls...")
    print("-" * 60)

    results = []
    for text in SAMPLE_TEXTS:
        try:
            result = analyze_sentiment(client, text)
            results.append(result)
            print(f"Text: {text[:60]}...")
            print(f"  Sentiment : {result['sentiment'].upper()}")
            print(f"  Positive  : {result['positive_score']:.2f}")
            print(f"  Negative  : {result['negative_score']:.2f}")
            print(f"  Neutral   : {result['neutral_score']:.2f}")
            print()
        except requests.HTTPError as exc:
            print(f"API call failed: {exc}")
            print(f"Response: {exc.response.text[:500]}")
else:
    print("Credentials not configured — skipping live API calls.")
    print("Configure AZURE_TENANT_ID, AZURE_CLIENT_ID, AZURE_CLIENT_SECRET,")
    print("and POWER_PLATFORM_ENV_URL in your .env file to run live calls.")

    # Simulate what the output would look like
    print()
    print("Simulated output (structure only, not real predictions):")
    simulated_results = [
        {"sentiment": "positive", "positive_score": 0.91, "negative_score": 0.02, "neutral_score": 0.07, "mixed_score": 0.00},
        {"sentiment": "negative", "positive_score": 0.05, "negative_score": 0.88, "neutral_score": 0.07, "mixed_score": 0.00},
        {"sentiment": "neutral",  "positive_score": 0.10, "negative_score": 0.05, "neutral_score": 0.85, "mixed_score": 0.00},
        {"sentiment": "negative", "positive_score": 0.03, "negative_score": 0.93, "neutral_score": 0.04, "mixed_score": 0.00},
    ]
    for text, sim in zip(SAMPLE_TEXTS, simulated_results):
        print(f"Text: {text[:60]}...")
        print(f"  Sentiment : {sim['sentiment'].upper()}")
        print(f"  Positive  : {sim['positive_score']:.2f}")
        print(f"  Negative  : {sim['negative_score']:.2f}")
        print(f"  Neutral   : {sim['neutral_score']:.2f}")
        print()

## 4. Document Processing: Invoice Extraction

The invoice extraction endpoint accepts a base64-encoded document and returns structured field data.

### Endpoint
```
POST {env_url}/api/data/v9.2/msdyn_PredictInvoice
```

### Request Body
```json
{
  "msdyn_DocumentBase64": "<base64 encoded PDF bytes>",
  "msdyn_MimeType": "application/pdf"
}
```

### Response Structure (abbreviated)
```json
{
  "msdyn_VendorName": "Contoso Supplies Ltd",
  "msdyn_VendorAddress": "123 Main St, Seattle, WA 98101",
  "msdyn_InvoiceId": "INV-2024-0891",
  "msdyn_InvoiceDate": "2024-02-15",
  "msdyn_DueDate": "2024-03-15",
  "msdyn_InvoiceTotal": 4250.00,
  "msdyn_SubTotal": 3900.00,
  "msdyn_TotalTax": 350.00,
  "msdyn_LineItems": [
    {"msdyn_Description": "Consulting services - January", "msdyn_Quantity": 20, "msdyn_UnitPrice": 195.00, "msdyn_Amount": 3900.00}
  ],
  "msdyn_ConfidenceScore": 0.94
}
```

The `msdyn_ConfidenceScore` is a model-level confidence for the overall extraction.
Individual fields may also carry their own confidence values in the full response.

In [ ]:
# Why: Document processing takes a binary file and returns structured data.
# The base64 encoding step is the key difference from text-based model calls.

def encode_file_to_base64(file_path: str) -> str:
    """
    Read a file from disk and encode it as a base64 string.

    Parameters
    ----------
    file_path : str
        Absolute or relative path to the file (PDF, JPEG, PNG, etc.)

    Returns
    -------
    str
        Base64-encoded file content
    """
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def extract_invoice_data(client: PowerPlatformClient, file_path: str) -> dict:
    """
    Extract structured data from an invoice document using AI Builder.

    Parameters
    ----------
    client : PowerPlatformClient
        Authenticated Power Platform client
    file_path : str
        Path to invoice file (PDF, JPEG, PNG, TIFF, BMP)

    Returns
    -------
    dict
        Normalized invoice fields: vendor_name, invoice_number, invoice_date,
        due_date, subtotal, tax, total, line_items, confidence, raw_response
    """
    # Infer MIME type from file extension
    ext = os.path.splitext(file_path)[1].lower()
    mime_types = {
        ".pdf": "application/pdf",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".tiff": "image/tiff",
        ".tif": "image/tiff",
        ".bmp": "image/bmp",
    }
    mime_type = mime_types.get(ext, "application/octet-stream")

    encoded_content = encode_file_to_base64(file_path)

    payload = {
        "msdyn_DocumentBase64": encoded_content,
        "msdyn_MimeType": mime_type,
    }

    response = client.post("msdyn_PredictInvoice", payload)
    response.raise_for_status()

    raw = response.json()

    # Normalize: strip msdyn_ prefix and rename to clean field names
    return {
        "vendor_name": raw.get("msdyn_VendorName"),
        "vendor_address": raw.get("msdyn_VendorAddress"),
        "invoice_number": raw.get("msdyn_InvoiceId"),
        "invoice_date": raw.get("msdyn_InvoiceDate"),
        "due_date": raw.get("msdyn_DueDate"),
        "subtotal": raw.get("msdyn_SubTotal"),
        "tax": raw.get("msdyn_TotalTax"),
        "total": raw.get("msdyn_InvoiceTotal"),
        "line_items": raw.get("msdyn_LineItems", []),
        "confidence": raw.get("msdyn_ConfidenceScore"),
        "raw_response": raw,
    }


def parse_line_items(line_items: list) -> list:
    """
    Normalize the line items array from the invoice extraction response.

    Parameters
    ----------
    line_items : list
        Raw line item array from invoice extraction response

    Returns
    -------
    list of dict
        Each dict has: description, quantity, unit_price, amount
    """
    return [
        {
            "description": item.get("msdyn_Description") or item.get("description"),
            "quantity": item.get("msdyn_Quantity") or item.get("quantity"),
            "unit_price": item.get("msdyn_UnitPrice") or item.get("unit_price"),
            "amount": item.get("msdyn_Amount") or item.get("amount"),
        }
        for item in line_items
    ]


print("Invoice extraction functions defined.")
print("Call extract_invoice_data(client, '/path/to/invoice.pdf') to process a real invoice.")

In [ ]:
# Demonstrate parsing the response structure without a live API call.
# This is what the normalize function produces from the raw API response.

SAMPLE_INVOICE_RESPONSE = {
    "msdyn_VendorName": "Contoso Supplies Ltd",
    "msdyn_VendorAddress": "123 Main St, Seattle, WA 98101",
    "msdyn_InvoiceId": "INV-2024-0891",
    "msdyn_InvoiceDate": "2024-02-15",
    "msdyn_DueDate": "2024-03-15",
    "msdyn_InvoiceTotal": 4250.00,
    "msdyn_SubTotal": 3900.00,
    "msdyn_TotalTax": 350.00,
    "msdyn_ConfidenceScore": 0.94,
    "msdyn_LineItems": [
        {
            "msdyn_Description": "Consulting services - January 2024",
            "msdyn_Quantity": 20,
            "msdyn_UnitPrice": 195.00,
            "msdyn_Amount": 3900.00,
        }
    ],
}

# Simulate normalize() output without making a live call
normalized = {
    "vendor_name": SAMPLE_INVOICE_RESPONSE["msdyn_VendorName"],
    "vendor_address": SAMPLE_INVOICE_RESPONSE["msdyn_VendorAddress"],
    "invoice_number": SAMPLE_INVOICE_RESPONSE["msdyn_InvoiceId"],
    "invoice_date": SAMPLE_INVOICE_RESPONSE["msdyn_InvoiceDate"],
    "due_date": SAMPLE_INVOICE_RESPONSE["msdyn_DueDate"],
    "subtotal": SAMPLE_INVOICE_RESPONSE["msdyn_SubTotal"],
    "tax": SAMPLE_INVOICE_RESPONSE["msdyn_TotalTax"],
    "total": SAMPLE_INVOICE_RESPONSE["msdyn_InvoiceTotal"],
    "confidence": SAMPLE_INVOICE_RESPONSE["msdyn_ConfidenceScore"],
    "line_items": parse_line_items(SAMPLE_INVOICE_RESPONSE["msdyn_LineItems"]),
}

print("Normalized invoice extraction output:")
print("-" * 50)
for field, value in normalized.items():
    if field != "line_items":
        print(f"{field:<20}: {value}")

print()
print("Line items:")
for i, item in enumerate(normalized["line_items"], 1):
    print(f"  Item {i}:")
    for k, v in item.items():
        print(f"    {k:<15}: {v}")

## 5. Comparing AI Builder with Direct Azure AI Services

AI Builder is built on top of Azure AI Services (formerly Cognitive Services). Understanding the
relationship helps you choose the right tool:

| Dimension | AI Builder | Azure AI Services Direct |
|---|---|---|
| **Integration** | Native in Power Automate, no code | Requires custom connector or Azure Function |
| **Authentication** | Power Platform credentials | Azure resource key or Managed Identity |
| **Pricing** | AI Builder credits | Per-transaction Azure pricing |
| **Data governance** | Within Power Platform boundary | Azure resource region/network |
| **Custom training** | Limited (custom document, object) | Full model fine-tuning available |
| **Latency** | Slightly higher (Platform overhead) | Lower (direct service call) |
| **Best for** | Power Automate flows, citizen developers | Azure-native apps, high volume, custom models |

The same underlying models power both surfaces. A sentiment analysis call through AI Builder
and a call through Azure Language Service `analyze-sentiment` endpoint use the same model family.
The differences are routing, authentication, and pricing.

In [ ]:
# Why: Seeing the Azure AI Services equivalent call side-by-side with the AI Builder call
# makes the abstraction layer concrete and helps you choose the right approach.

def analyze_sentiment_azure_direct(
    text: str,
    endpoint: str,
    api_key: str,
) -> dict:
    """
    Analyze sentiment via the Azure Language Service API directly.

    This is the underlying service that AI Builder uses internally.
    Call this directly when you need lower latency, higher volume,
    or integration outside the Power Platform.

    Parameters
    ----------
    text : str
        Text to analyze
    endpoint : str
        Azure Language Service endpoint URL
        e.g. https://your-resource.cognitiveservices.azure.com
    api_key : str
        Azure Language Service API key

    Returns
    -------
    dict
        Normalized result matching the AI Builder response structure
    """
    # Azure Language Service uses a batched request format
    url = f"{endpoint.rstrip('/')}/language/:analyze-text?api-version=2023-04-01"
    payload = {
        "kind": "SentimentAnalysis",
        "parameters": {"modelVersion": "latest"},
        "analysisInput": {
            "documents": [{"id": "1", "language": "en", "text": text}]
        },
    }
    headers = {
        "Ocp-Apim-Subscription-Key": api_key,
        "Content-Type": "application/json",
    }

    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()

    data = response.json()
    doc = data["results"]["documents"][0]
    scores = doc["confidenceScores"]

    # Normalize to match AI Builder response structure
    return {
        "sentiment": doc["sentiment"],
        "positive_score": scores["positive"],
        "negative_score": scores["negative"],
        "neutral_score": scores["neutral"],
        "mixed_score": 0.0,  # Azure Language Service doesn't return mixed score
        "raw_response": data,
    }


# Show the structural difference between the two API call shapes
print("AI Builder request body:")
print(json.dumps(
    {"msdyn_Text": "<text>", "msdyn_LanguageCode": "en"},
    indent=2,
))
print()

print("Azure Language Service request body (equivalent):")
print(json.dumps({
    "kind": "SentimentAnalysis",
    "parameters": {"modelVersion": "latest"},
    "analysisInput": {
        "documents": [{"id": "1", "language": "en", "text": "<text>"}]
    },
}, indent=2))
print()

print("Key difference: AI Builder uses simple field names and single-document calls.")
print("Azure Language Service uses a batched document array and operation type parameter.")
print("Both return equivalent confidence scores in different JSON structures.")

## 6. Exercises

Apply what you've learned by completing these exercises. Each exercise has an assertion-based
self-check. You do not need live API credentials — work with the response parsing logic.

In [ ]:
# --- Exercise 6.1 ---
# Write a function that takes an analyze_sentiment result dict and returns
# a routing decision for a support ticket system.
#
# Routing rules:
#   - negative_score >= 0.7 -> route to "escalation"
#   - positive_score >= 0.6 -> route to "standard"
#   - all others           -> route to "review"
#
# The function should return a string: "escalation", "standard", or "review"

def route_ticket_by_sentiment(sentiment_result: dict) -> str:
    """
    Route a support ticket based on AI Builder sentiment analysis output.

    Parameters
    ----------
    sentiment_result : dict
        Output from analyze_sentiment() with keys:
        sentiment, positive_score, negative_score, neutral_score, mixed_score

    Returns
    -------
    str
        One of: "escalation", "standard", "review"
    """
    # TODO: Implement routing logic using the rules above
    pass


# Self-check
def test_exercise_6_1():
    high_negative = {"sentiment": "negative", "positive_score": 0.05,
                     "negative_score": 0.88, "neutral_score": 0.07, "mixed_score": 0.00}
    high_positive = {"sentiment": "positive", "positive_score": 0.91,
                     "negative_score": 0.02, "neutral_score": 0.07, "mixed_score": 0.00}
    neutral_mixed = {"sentiment": "neutral", "positive_score": 0.30,
                     "negative_score": 0.35, "neutral_score": 0.35, "mixed_score": 0.00}

    result_neg = route_ticket_by_sentiment(high_negative)
    assert result_neg == "escalation", (
        f"Expected 'escalation' for negative_score=0.88, got '{result_neg}'. "
        "Check: is negative_score >= 0.7?"
    )

    result_pos = route_ticket_by_sentiment(high_positive)
    assert result_pos == "standard", (
        f"Expected 'standard' for positive_score=0.91, got '{result_pos}'. "
        "Check: is positive_score >= 0.6 and negative_score < 0.7?"
    )

    result_neutral = route_ticket_by_sentiment(neutral_mixed)
    assert result_neutral == "review", (
        f"Expected 'review' for ambiguous scores, got '{result_neutral}'. "
        "Check: neither negative >= 0.7 nor positive >= 0.6."
    )

    print("Exercise 6.1 passed: ticket routing logic is correct.")


test_exercise_6_1()

In [ ]:
# --- Exercise 6.2 ---
# Write a function that validates an invoice extraction result.
# A valid result must have all of these fields non-None and non-empty:
#   vendor_name, invoice_number, invoice_date, total
# AND total must be a positive number (> 0).
# AND confidence must be >= 0.7 (minimum acceptable confidence threshold).
#
# Return a tuple: (is_valid: bool, issues: list of str)
# If valid, issues should be an empty list.
# If invalid, issues should list each problem as a human-readable string.

def validate_invoice_extraction(invoice: dict) -> tuple:
    """
    Validate an AI Builder invoice extraction result for completeness and quality.

    Parameters
    ----------
    invoice : dict
        Output from extract_invoice_data() with keys:
        vendor_name, invoice_number, invoice_date, due_date,
        subtotal, tax, total, line_items, confidence

    Returns
    -------
    tuple of (bool, list of str)
        (True, []) if valid
        (False, ["issue 1", "issue 2", ...]) if invalid
    """
    # TODO: Implement validation logic
    pass


# Self-check
def test_exercise_6_2():
    good_invoice = {
        "vendor_name": "Contoso Ltd",
        "invoice_number": "INV-001",
        "invoice_date": "2024-02-15",
        "due_date": "2024-03-15",
        "total": 4250.00,
        "subtotal": 3900.00,
        "tax": 350.00,
        "line_items": [],
        "confidence": 0.94,
    }
    is_valid, issues = validate_invoice_extraction(good_invoice)
    assert is_valid is True, f"Good invoice should be valid, got issues: {issues}"
    assert issues == [], f"Good invoice should have no issues, got: {issues}"

    missing_vendor = dict(good_invoice, vendor_name=None)
    is_valid, issues = validate_invoice_extraction(missing_vendor)
    assert is_valid is False, "Invoice with null vendor_name should be invalid"
    assert any("vendor" in issue.lower() for issue in issues), (
        f"Issues should mention vendor_name, got: {issues}"
    )

    low_confidence = dict(good_invoice, confidence=0.55)
    is_valid, issues = validate_invoice_extraction(low_confidence)
    assert is_valid is False, "Invoice with confidence=0.55 (below 0.7) should be invalid"
    assert any("confidence" in issue.lower() for issue in issues), (
        f"Issues should mention confidence threshold, got: {issues}"
    )

    zero_total = dict(good_invoice, total=0)
    is_valid, issues = validate_invoice_extraction(zero_total)
    assert is_valid is False, "Invoice with total=0 should be invalid"

    print("Exercise 6.2 passed: invoice validation logic is correct.")


test_exercise_6_2()

In [ ]:
# --- Exercise 6.3 ---
# Write a function that constructs a Power Automate-compatible payload
# for the AI Builder sentiment analysis action.
#
# The function takes a plain text string and an optional language code,
# and returns the dict that would be sent as the request body.
#
# Rules:
#   - If text is empty or None, raise ValueError with message:
#     "Text cannot be empty"
#   - If language is provided, include it as "msdyn_LanguageCode"
#   - If language is None or empty string, omit the language key entirely
#   - Always include "msdyn_Text"

def build_sentiment_payload(text: str, language: str | None = None) -> dict:
    """
    Build the request payload for the AI Builder sentiment analysis API.

    Parameters
    ----------
    text : str
        Text to analyze
    language : str or None
        BCP-47 language code (e.g. 'en', 'fr', 'de').
        If None or empty string, omit from payload for auto-detection.

    Returns
    -------
    dict
        Request body with 'msdyn_Text' and optionally 'msdyn_LanguageCode'

    Raises
    ------
    ValueError
        If text is empty or None
    """
    # TODO: Implement this function
    pass


# Self-check
def test_exercise_6_3():
    # With language
    payload = build_sentiment_payload("Hello world", "en")
    assert payload.get("msdyn_Text") == "Hello world", (
        f"Expected msdyn_Text='Hello world', got: {payload.get('msdyn_Text')}"
    )
    assert payload.get("msdyn_LanguageCode") == "en", (
        f"Expected msdyn_LanguageCode='en', got: {payload.get('msdyn_LanguageCode')}"
    )

    # Without language — key should be absent
    payload_no_lang = build_sentiment_payload("Hello world")
    assert "msdyn_LanguageCode" not in payload_no_lang, (
        "When language is None, msdyn_LanguageCode should not appear in the payload"
    )

    # Empty string language — also omit
    payload_empty_lang = build_sentiment_payload("Hello world", "")
    assert "msdyn_LanguageCode" not in payload_empty_lang, (
        "When language is empty string, msdyn_LanguageCode should not appear"
    )

    # Empty text should raise ValueError
    raised = False
    try:
        build_sentiment_payload("")
    except ValueError as exc:
        raised = True
        assert "empty" in str(exc).lower(), (
            f"ValueError message should mention 'empty', got: {exc}"
        )
    assert raised, "Empty text should raise ValueError"

    print("Exercise 6.3 passed: payload builder logic is correct.")


test_exercise_6_3()

## Solutions

Reference implementations for the exercises. Attempt each exercise before looking.

In [ ]:
# SOLUTION 6.1: Ticket routing by sentiment

def route_ticket_by_sentiment_solution(sentiment_result: dict) -> str:
    if sentiment_result["negative_score"] >= 0.7:
        return "escalation"
    if sentiment_result["positive_score"] >= 0.6:
        return "standard"
    return "review"


# SOLUTION 6.2: Invoice validation

def validate_invoice_extraction_solution(invoice: dict) -> tuple:
    issues = []

    required_fields = ["vendor_name", "invoice_number", "invoice_date", "total"]
    for field in required_fields:
        value = invoice.get(field)
        if value is None or (isinstance(value, str) and not value.strip()):
            issues.append(f"Missing required field: {field}")

    total = invoice.get("total")
    if total is not None and total <= 0:
        issues.append(f"Invoice total must be positive, got: {total}")

    confidence = invoice.get("confidence")
    if confidence is not None and confidence < 0.7:
        issues.append(
            f"Confidence score {confidence:.2f} is below the minimum threshold of 0.70 — "
            "manual review required"
        )

    return (len(issues) == 0, issues)


# SOLUTION 6.3: Payload builder

def build_sentiment_payload_solution(text: str, language: str | None = None) -> dict:
    if not text:
        raise ValueError("Text cannot be empty")

    payload = {"msdyn_Text": text}
    if language:  # truthy check: None and empty string both excluded
        payload["msdyn_LanguageCode"] = language
    return payload


print("Solutions loaded. Verify your implementations match the logic above.")

In [ ]:
section_divider("Summary")

## Summary

**Key Takeaways from this Notebook:**

1. **Authentication Pattern**: Power Platform APIs use Azure AD client credentials. Cache tokens
   and refresh before expiry to avoid unnecessary round trips.

2. **AI Builder Endpoint Naming**: All AI Builder fields use the `msdyn_` prefix in the API.
   Normalizing this prefix in your client layer produces cleaner downstream code.

3. **Document Processing**: Files must be base64-encoded before submission. Infer MIME type
   from file extension for correct parsing at the server.

4. **Response Validation**: Always validate confidence scores and required field presence
   before using AI Builder output in downstream automation logic.

5. **AI Builder vs Azure Direct**: AI Builder is simpler to integrate into Power Platform flows.
   Azure AI Services direct gives lower latency and more control for high-volume or
   Azure-native applications.

**What's Next:**
- Module 08 Exercise: Build effective Copilot prompts for real business scenarios
- Module 09: Copilot agents — autonomous multi-step AI workflows

---

**Additional References:**
- [AI Builder documentation](https://learn.microsoft.com/en-us/ai-builder/overview)
- [Power Platform REST API reference](https://learn.microsoft.com/en-us/power-apps/developer/data-platform/webapi/overview)
- [Azure AI Language Service documentation](https://learn.microsoft.com/en-us/azure/ai-services/language-service/)

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])